# Lesson 3.3 — ADQL Aggregations and Time Series

## Learning Objectives
- Use ADQL aggregate functions: `count()`, `avg()`, `max()`, `min()`, `sum()`, `distinct()`
- Group results with implicit GROUP BY (from select fields)
- Use the `series()` function to produce time-series output for charts
- Understand when to use aggregation vs. raw record queries

**Reference**: [ADQL Reference — Aggregations](https://help.splunk.com/en/appdynamics-saas/analytics/26.2.0/analytics/adql-reference/adql-queries) | [Grafana Analytics Examples](https://grafana.com/docs/plugins/dlopes7-appdynamics-datasource/latest/query-editor/#analytics-query-examples)

## 1. Aggregate Functions

ADQL supports the standard SQL aggregate functions:

| Function | Description | Example |
|----------|-------------|----------|
| `count(*)` | Count all records | `SELECT count(*) FROM transactions` |
| `count(field)` | Count non-null values | `SELECT count(userId) FROM transactions` |
| `avg(field)` | Average value | `SELECT avg(responseTime) FROM transactions` |
| `max(field)` | Maximum value | `SELECT max(responseTime) FROM transactions` |
| `min(field)` | Minimum value | `SELECT min(responseTime) FROM transactions` |
| `sum(field)` | Sum of values | `SELECT sum(responseTime) FROM transactions` |
| `distinct(field)` | Unique values | `SELECT distinct(transactionName) FROM transactions` |
| `percentile(field, n)` | nth percentile | `SELECT percentile(responseTime, 95) FROM transactions` |

### GROUP BY — implicit in ADQL:
When you SELECT a non-aggregated field alongside an aggregate, ADQL implicitly groups by that field:

```sql
-- Groups by transactionName automatically
SELECT transactionName, count(*), avg(responseTime) FROM transactions
```

This is equivalent to:
```sql
-- Standard SQL equivalent
SELECT transactionName, count(*), avg(responseTime) FROM transactions GROUP BY transactionName
```

In [ ]:
# Rebuild dataset from Lesson 3.2
import random
from datetime import datetime, timedelta
from collections import defaultdict
import statistics

random.seed(42)

def generate_transaction_dataset(n=500):
    records = []
    transactions = ["/checkout", "/search", "/cart/add", "/cart/view", "/login", "/product/{id}"]
    tiers = ["WebFrontend", "MobileAPI"]
    base_ts = int((datetime.now() - timedelta(minutes=30)).timestamp() * 1000)
    for i in range(n):
        tx = random.choice(transactions)
        ue = random.choices(["NORMAL","SLOW","VERY_SLOW","ERROR"], weights=[82,10,5,3])[0]
        rt_map = {"NORMAL": random.gauss(280,60), "SLOW": random.gauss(900,100),
                  "VERY_SLOW": random.gauss(2100,300), "ERROR": random.gauss(500,200)}
        records.append({
            "eventTimestamp": base_ts + i * 3600,
            "transactionName": tx, "tierName": random.choice(tiers),
            "applicationName": "StoreFront",
            "responseTime": max(50, int(rt_map[ue])),
            "userExperience": ue, "errorOccurred": ue == "ERROR",
            "userId": str(random.randint(1000, 9999)),
            "country": random.choice(["United States","Germany","Japan","Brazil","United Kingdom"]),
            "browser": random.choice(["Chrome","Firefox","Safari","Edge"]),
            "deviceType": random.choice(["Desktop","Mobile","Tablet"]),
        })
    return records

DATA = generate_transaction_dataset(500)
print(f"Dataset ready: {len(DATA)} records")

In [ ]:
# ADQL Query: Transaction performance summary grouped by transaction name
# ADQL:
# SELECT transactionName, count(*) AS total, avg(responseTime) AS avgRT,
#        max(responseTime) AS maxRT
# FROM transactions
# ORDER BY total DESC

# Python equivalent (GROUP BY transactionName)
grouped = defaultdict(list)
for r in DATA:
    grouped[r['transactionName']].append(r['responseTime'])

results = []
for tx, times in grouped.items():
    results.append({
        "transactionName": tx,
        "total":  len(times),
        "avgRT":  statistics.mean(times),
        "maxRT":  max(times),
        "p95RT":  sorted(times)[int(len(times) * 0.95)]
    })

results.sort(key=lambda r: r['total'], reverse=True)

print("ADQL: SELECT transactionName, count(*) AS total, avg(responseTime) AS avgRT, max(responseTime) AS maxRT")
print("      FROM transactions ORDER BY total DESC")
print()
print(f"{'Transaction':<22} {'Count':>7} {'Avg RT':>9} {'Max RT':>9} {'P95 RT':>9}")
print("-" * 62)
for r in results:
    print(f"{r['transactionName']:<22} {r['total']:>7} {r['avgRT']:>8.1f}ms {r['maxRT']:>8}ms {r['p95RT']:>8}ms")

In [ ]:
# ADQL Query: Count and rank by user experience type
# ADQL:
# SELECT transactionName, userExperience, count(*) AS cnt
# FROM transactions
# ORDER BY cnt DESC

ue_grouped = defaultdict(int)
for r in DATA:
    key = (r['transactionName'], r['userExperience'])
    ue_grouped[key] += 1

ue_results = sorted([
    {"transactionName": k[0], "userExperience": k[1], "cnt": v}
    for k, v in ue_grouped.items()
], key=lambda r: r['cnt'], reverse=True)

print("ADQL: SELECT transactionName, userExperience, count(*) AS cnt")
print("      FROM transactions ORDER BY cnt DESC")
print()
ue_icons = {"NORMAL": "✅", "SLOW": "🟡", "VERY_SLOW": "🔴", "ERROR": "💥"}
for r in ue_results[:12]:
    icon = ue_icons.get(r['userExperience'], "")
    print(f"  {r['transactionName']:<22} {icon} {r['userExperience']:<12} : {r['cnt']} calls")

## 2. The `series()` Function — Time-Series Output

The `series()` function is ADQL's key tool for producing **time-series data** suitable for time-series chart widgets.

### Syntax:
```sql
SELECT series(eventTimestamp, '<interval>'), <optional_group_field>, <aggregate>
FROM transactions
```

### Interval options:
| Interval | String |
|----------|--------|
| 1 minute | `'1m'` |
| 5 minutes | `'5m'` |
| 10 minutes | `'10m'` |
| 1 hour | `'1h'` |

### Examples from official docs:

```sql
-- Transaction count per minute
SELECT series(eventTimestamp, '1m'), count(*) FROM transactions

-- Average response time per 5 minutes, grouped by user experience
SELECT series(eventTimestamp, '5m'), userExperience, count(*) FROM transactions

-- Error count per minute for /checkout
SELECT series(eventTimestamp, '1m'), count(*) FROM transactions
WHERE transactionName = '/checkout' AND errorOccurred = true
```

> **Important**: The `series()` function is **only available in Dash Studio's Time Series widget** and in the Grafana Analytics query type. It cannot be used in the regular Analytics search UI.

**Reference**: [ADQL Queries in Dash Studio](https://help.splunk.com/en/appdynamics-saas/get-started/26.2.0/dashboards-and-reports/dash-studio/data-binding/adql-queries)

In [ ]:
# Simulate the series() function: bucket records into 1-minute intervals
import matplotlib.pyplot as plt

def simulate_series(data, interval_ms, group_field=None, aggregate='count'):
    """
    Simulate the ADQL series() function.
    Buckets data into time windows and computes aggregate.
    """
    if not data:
        return {}
    
    min_ts = min(r['eventTimestamp'] for r in data)
    max_ts = max(r['eventTimestamp'] for r in data)
    
    buckets = defaultdict(lambda: defaultdict(list))
    for r in data:
        bucket_ts = (r['eventTimestamp'] - min_ts) // interval_ms * interval_ms + min_ts
        key = r.get(group_field, 'ALL') if group_field else 'ALL'
        buckets[bucket_ts][key].append(r['responseTime'])
    
    result = {}
    all_keys = set()
    for bucket_ts, groups in buckets.items():
        result[bucket_ts] = {}
        for key, values in groups.items():
            all_keys.add(key)
            if aggregate == 'count':
                result[bucket_ts][key] = len(values)
            elif aggregate == 'avg':
                result[bucket_ts][key] = statistics.mean(values)
            elif aggregate == 'max':
                result[bucket_ts][key] = max(values)
    return result, sorted(all_keys)

ONE_MINUTE_MS = 60 * 1000

# Simulate: SELECT series(eventTimestamp, '1m'), userExperience, count(*) FROM transactions
series_data, ue_groups = simulate_series(DATA, ONE_MINUTE_MS, group_field='userExperience', aggregate='count')
timestamps = sorted(series_data.keys())

# Plot stacked bar chart (like a Dash Studio Time Series widget would show)
import numpy as np
ue_colors = {"NORMAL": "#4CAF50", "SLOW": "#FFC107", "VERY_SLOW": "#F44336", "ERROR": "#9C27B0"}
ue_order = ["NORMAL", "SLOW", "VERY_SLOW", "ERROR"]

fig, ax = plt.subplots(figsize=(13, 5))
bottom = np.zeros(len(timestamps))
x_labels = [f"T+{(ts - timestamps[0]) // ONE_MINUTE_MS}m" for ts in timestamps]

for ue in ue_order:
    values = [series_data[ts].get(ue, 0) for ts in timestamps]
    ax.bar(range(len(timestamps)), values, bottom=bottom,
           label=ue, color=ue_colors.get(ue, "gray"), alpha=0.85)
    bottom += np.array(values)

ax.set_xticks(range(len(timestamps)))
ax.set_xticklabels(x_labels, rotation=45, fontsize=8)
ax.set_xlabel("Time (1-min buckets)")
ax.set_ylabel("Transaction Count")
ax.set_title("ADQL: SELECT series(eventTimestamp,'1m'), userExperience, count(*) FROM transactions")
ax.legend()
plt.tight_layout()
plt.savefig('adql_series.png', dpi=100, bbox_inches='tight')
plt.show()
print("This is what the Dash Studio Time Series widget renders from an ADQL series() query.")

In [ ]:
# Ready-to-paste ADQL queries for Dash Studio and Grafana

adql_examples = [
    {
        "description": "Transaction count over time, grouped by user experience (stacked chart)",
        "query": """SELECT series(eventTimestamp, '1m'), userExperience, count(*)
FROM transactions"""
    },
    {
        "description": "Average response time per minute per transaction name",
        "query": """SELECT series(eventTimestamp, '1m'), transactionName, avg(responseTime) AS avgRT
FROM transactions"""
    },
    {
        "description": "Error count per minute (for error rate chart)",
        "query": """SELECT series(eventTimestamp, '1m'), count(*) AS errors
FROM transactions
WHERE errorOccurred = true"""
    },
    {
        "description": "Slowest transactions overall (aggregated)",
        "query": """SELECT transactionName, count(*) AS total, avg(responseTime) AS avgResponseTime
FROM transactions
ORDER BY avgResponseTime DESC
LIMIT 10"""
    },
    {
        "description": "Count errors grouped by transaction name (for error breakdown)",
        "query": """SELECT transactionName, count(*) AS errorCount
FROM transactions
WHERE errorOccurred = true
ORDER BY errorCount DESC"""
    },
]

print("Ready-to-use ADQL Queries:")
print("=" * 70)
for ex in adql_examples:
    print(f"\n-- {ex['description']}")
    print(ex['query'])

In [ ]:
# EXERCISE 3.3: Build an aggregation query
#
# Business Question: "Which countries have the highest error rates?"
#
# TASK: Write the ADQL query as a comment, then implement the Python equivalent.
# Expected output:
#   country, total_calls, error_count, error_rate%
#   sorted by error_rate descending

# ADQL:
# SELECT ??? FROM transactions ???

country_stats = defaultdict(lambda: {"total": 0, "errors": 0})

# TODO: Populate country_stats from DATA

if any(v['total'] > 0 for v in country_stats.values()):
    country_results = [
        {
            "country": c,
            "total": v['total'],
            "errors": v['errors'],
            "error_rate": v['errors'] / v['total'] * 100 if v['total'] > 0 else 0
        }
        for c, v in country_stats.items()
    ]
    country_results.sort(key=lambda r: r['error_rate'], reverse=True)
    
    print(f"{'Country':<20} {'Total':>8} {'Errors':>8} {'Error Rate':>12}")
    print("-" * 52)
    for r in country_results:
        print(f"{r['country']:<20} {r['total']:>8} {r['errors']:>8} {r['error_rate']:>12.2f}%")
else:
    print("Populate country_stats from DATA to see the results.")

## Summary

- Aggregate functions: `count(*)`, `avg()`, `max()`, `min()`, `sum()`, `distinct()`, `percentile()`
- ADQL groups automatically when non-aggregated fields appear alongside aggregates
- The `series(eventTimestamp, '1m')` function buckets data into time windows for time-series charts
- `series()` only works in Dash Studio Time Series widgets and Grafana Analytics queries
- Use aggregation queries for dashboards; use raw SELECT queries for troubleshooting

---

**Next**: [Lesson 3.4 — ADQL Analytics Tables](lesson-3.4-adql-analytics-tables.ipynb)